In [ ]:
def rt2xy(r,theta):
    return r*np.cos(theta),r*np.sin(theta) #cartesian positions of points on which i have the field B_func 
sindex=25
s_fixed=ss[sindex]
nk=4 #maximum abs of k
def ByiBx(x, y):#analytical definition of By + i*Bx at s=s_fixed, needs ndarrays comng from bpmeth field expansion
    return By_func(x, y, s_fixed) + 1j * Bx_func(x, y, s_fixed)
rr=np.linspace(rmin,rmax,nr)

def dkharmonics(ByiBx,rr=rr,nk=nk,ntheta=102):
  '''
  returns dk as function of r 
  ''' 
   #radial positions
  out=np.empty((len(rr),2*nk+1),dtype=complex) #prepare to store dk's as functions of r0 
  theta=np.arange(ntheta)/ntheta*2*np.pi
  for ir,r in enumerate(rr):
    x,y=rt2xy(r,theta)
    b=ByiBx(x,y)
    d=np.fft.fft(b)/ntheta #discrete fourier coefficients: dk at this r0
    out[ir,0]=d[0]#k=0
    out[ir,1::2]=d[1:nk+1] #first "fourier frequencies" are for positive k, stored in odd indexes
    out[ir,2::2]=d[ntheta:ntheta-nk-1:-1]#the last ones are for negative frequencies (take them in reverse order), stored in even indexes
  return out

def dklfit(dkofrarray,rr,nk, nl):
  '''
  takes the result from dkharmonics and fits d_k/r^|k| for different k values as a funtion of r^2l, 
  the coefficients of the fit are the values d_kl 
  parameters:
  rr=array of r values
  nk= max of |k| (not the number of different k values)
  nl= number of l values
  returns: matrix dkl
  '''
  polynomial=np.zeros((len(rr), 2*nk+1),dtype=complex)#to store d_k/r^|k|
  ka=np.linspace(-nk,nk,2*nk+1)
  for ir,r in enumerate(rr):
     polynomial[ir,0]=dkofrarray[ir,0]#for k=0 it would be divided by r0^0=1
     polynomial[ir,1::2]=dkofrarray[ir,1::2]/r**ka[nk+1 : 2*nk+1] #for positive k, odd indexes
     polynomial[ir,2::2]=dkofrarray[ir,2::2]/r**-ka[0: nk] #for negative k
  dkl=np.zeros((2*nk+1, nl), dtype=complex)#different k along rows, different l along columns
  dkl[0,:]=np.polyfit(rr**2,polynomial[:,0],deg=nl-1)[::-1] #k=0, fit r^2 since the exponent is 2l
  dkl[1::2,:]=np.polyfit(rr**2, polynomial[:,1::2], deg=nl-1)[::-1,].T #positive k
  dkl[2::2,:]=np.polyfit(rr**2, polynomial[:,2::2], deg=nl-1)[::-1,].T #negative k
  return dkl 

#now let's recover the a,b coefficients
def bnianHAopt(dkl, nl, nk):
    # Precompute factorials
    fact = np.array([factorial(n) for n in range(nk)], dtype=np.int16)

    # Build k index dict mapping: row index for k = 0,1,-1,2,-2,...
    k_index = {0: 0}
    for i in range(1, nk+1):
        k_index[i]  = 2*i - 1
        k_index[-i] = 2*i #negative k gets even row

    result = np.zeros(nk, dtype=np.complex128)

    for n in range(nk):
        s = 0.0 + 0.0j #initialize sum
        max_l = min(n//2, nl-1)

        for l in range(max_l + 1):
            k = n - 2*l
            if k == 0:
                s += dkl[k_index[0], l]
            else:
                s += dkl[k_index[k], l] + dkl[k_index[-k], l]

        result[n] = fact[n] * s

    return result
def mylocalharmonics():
    '''
    local harmonic analysis from By+iBx to dkl and coefficients a,b 
    '''
    dk=dkharmonics(ByiBx=ByiBx, rr=rr,nk=nk,ntheta=102)
    dkl=dklfit(dk,rr,nk, nl)
    an=bnianHAopt(dkl, nl,nk).imag
    bn=bnianHAopt(dkl, nl,nk).real
    return an, bn

In [ ]:
#timing
def mylocalharmonics():
    '''
    local harmonic analysis from By+iBx to dkl and coefficients a,b 
    '''
    dk=dkharmonics(ByiBx=ByiBx, rr=rr,nk=nk,ntheta=102)
    dkl=dklfit(dk,rr,nk, nl)
    an=bnianHAopt(dkl, nl,nk).imag
    bn=bnianHAopt(dkl, nl,nk).real
    return an, bn

def bpmethlocalharmonics():
    dkl=bpmeth.harmonics(ByiBx=ByiBx,nk=nk,rmin=rmin, rmax=rmax,nr=nr, ntheta=102)
    an=bpmeth.calc_coeffs(dkl).imag
    bn=bpmeth.calc_coeffs(dkl).real
    return an, bn
N=1000
import timeit
mytime=timeit.timeit("mylocalharmonics()", globals=globals(), number=N)
bpmethtime=timeit.timeit("bpmethlocalharmonics()", globals=globals(), number=N)
print("my time=", mytime)
print("bpmeth time", bpmethtime)
print("relative improvement timed over", N,"repetitions: (bpmeth time - new code time)/bpmeth time =", (bpmethtime-mytime)/bpmethtime*100,"%")

my time= 22.60738687400044
bpmeth time 24.8815770709989
relative improvement timed over 1000 repetitions: (bpmeth time - new code time)/bpmeth time = 9.140056478370004 %
